[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Bigdata-com/bigdata-cookbook/blob/main/API_Tutorials/Internal_Contents/upload_and_search.ipynb)

# Upload & Search Private Content

**Bigdata.com APIs** · [Enrich document](https://docs.bigdata.com/api-reference/documents/enrich-document) · [Search tag filter](https://docs.bigdata.com/api-reference/search/search-documents#body-query-filters-tag)

This notebook uploads two sample PDFs to Bigdata.com, waits for enrichment and indexing, then searches them using the `query.filters.tag` filter.

## Install dependencies

Run once to install `requests`. Skip if already installed.

In [1]:
%pip install -q requests
print("✓ Required packages installed.")

Note: you may need to restart the kernel to use updated packages.
✓ Required packages installed.


## Setup

Import libraries and configure API endpoints. Sample PDFs live in `sample_files/` next to this notebook.

In [2]:
import os
import time
from pathlib import Path

import requests

API_BASE_URL = "https://api.bigdata.com"
DOCUMENTS_ENDPOINT = f"{API_BASE_URL}/contents/v1/documents"
SEARCH_ENDPOINT = f"{API_BASE_URL}/v1/search"

NOTEBOOK_DIR = Path.cwd()
SAMPLE_DIR = NOTEBOOK_DIR / "sample_files"

print(f"Sample files directory: {SAMPLE_DIR}")

Sample files directory: /Users/bakulkumarkakadiya/dev/github/cookbook-master/bigdata-cookbook/API_Tutorials/Internal_Contents/sample_files


## API key

Load `BIGDATA_API_KEY` from the environment. On Google Colab, fall back to Colab Secrets.

In [3]:
API_KEY = os.getenv("BIGDATA_API_KEY")

if not API_KEY:
    try:
        from google.colab import userdata

        API_KEY = userdata.get("BIGDATA_API_KEY")
    except ImportError:
        pass

if not API_KEY:
    raise ValueError(
        "BIGDATA_API_KEY not found. Set it as an environment variable "
        "(export BIGDATA_API_KEY=your_key) or add it to Colab Secrets."
    )

session = requests.Session()
session.headers.update({"Content-Type": "application/json", "X-API-KEY": API_KEY})
print("✓ API key loaded successfully")

✓ API key loaded successfully


## Upload helpers

Three-step upload flow: request a pre-signed URL, PUT the file, then poll until enrichment completes.

In [4]:
def request_upload_url(
    file_name: str,
    tags: list[str],
    share_with_org: bool = False,
) -> dict:
    """Request a pre-signed upload URL and document id from the Content API."""
    payload = {
        "file_name": file_name,
        "tags": tags,
        "share_with_org": share_with_org,
    }
    response = session.post(DOCUMENTS_ENDPOINT, json=payload)
    response.raise_for_status()
    return response.json()


def upload_file_to_url(upload_url: str, file_path: Path) -> None:
    """PUT the raw file bytes to the pre-signed URL."""
    with file_path.open("rb") as file_handle:
        response = requests.put(upload_url, data=file_handle)
    response.raise_for_status()


def wait_for_document(
    document_id: str,
    poll_interval_seconds: int = 5,
    timeout_seconds: int = 900,
) -> dict:
    """Poll document metadata until status is completed or failed."""
    deadline = time.time() + timeout_seconds
    while time.time() < deadline:
        response = session.get(f"{DOCUMENTS_ENDPOINT}/{document_id}")
        response.raise_for_status()
        metadata = response.json()
        status = metadata.get("status")
        print(f"  status: {status}")
        if status == "completed":
            return metadata
        if status == "failed":
            raise RuntimeError(
                f"Document {document_id} failed: {metadata.get('error_code')}"
            )
        time.sleep(poll_interval_seconds)
    raise TimeoutError(
        f"Document {document_id} did not complete within {timeout_seconds}s"
    )


def upload_document(
    file_path: Path,
    file_name: str,
    tags: list[str],
) -> dict:
    """Upload a file and wait for enrichment to finish."""
    print(f"Uploading {file_path.name} as '{file_name}'...")
    upload_info = request_upload_url(file_name=file_name, tags=tags)
    document_id = upload_info["id"]
    print(f"  document id: {document_id}")
    upload_file_to_url(upload_info["url"], file_path)
    print("  file uploaded, waiting for enrichment...")
    return wait_for_document(document_id)

## Upload Boeing document

Upload `Boeing-Corporate-Actions.pdf` with custom file name and tags.

In [5]:
boeing_metadata = upload_document(
    file_path=SAMPLE_DIR / "Boeing-Corporate-Actions.pdf",
    file_name="Boeing Corporate Actions from 2024 to 2026",
    tags=["Boeing", "Corporate Actions", "Workflow"],
)
print(f"✓ Boeing document ready: {boeing_metadata['id']}")

Uploading Boeing-Corporate-Actions.pdf as 'Boeing Corporate Actions from 2024 to 2026'...
  document id: F7CF8E826A82B81A23433FF07F7CC2D1
  file uploaded, waiting for enrichment...
  status: placeholder
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing

RuntimeError: Document F7CF8E826A82B81A23433FF07F7CC2D1 failed: UAS_ERR_2004

## Upload Brazil document

Upload `Brazil_Economic_Analysis_2026.pdf` with custom file name and tags.

In [ ]:
brazil_metadata = upload_document(
    file_path=SAMPLE_DIR / "Brazil_Economic_Analysis_2026.pdf",
    file_name="Brazil Economic Analsis as of June 2026",
    tags=["Brazil", "Country", "Macro Analysis", "MCP"],
)
print(f"✓ Brazil document ready: {brazil_metadata['id']}")

## Search helper

Search uploaded files using `query.filters.tag` and restrict results to the `my_files` category.

In [ ]:
def search_by_tag(text: str, tag: str, max_chunks: int = 5) -> dict:
    """Run a fast-mode search filtered by tag within my_files."""
    payload = {
        "search_mode": "fast",
        "query": {
            "text": text,
            "auto_enrich_filters": False,
            "filters": {
                "category": {
                    "mode": "INCLUDE",
                    "values": ["my_files"],
                },
                "tag": {
                    "any_of": [tag],
                },
            },
            "max_chunks": max_chunks,
        },
    }
    response = session.post(SEARCH_ENDPOINT, json=payload)
    response.raise_for_status()
    return response.json()


def print_search_results(data: dict, max_results: int = 3) -> None:
    """Print a concise summary of search results."""
    results = data.get("results", [])
    print(f"Found {len(results)} documents")
    print(f"API units used: {data.get('usage', {}).get('api_query_units', 0)}\n")
    for index, doc in enumerate(results[:max_results], 1):
        print(f"--- Document {index} ---")
        print(f"Headline: {doc.get('headline', 'N/A')}")
        print(f"Document ID: {doc.get('id', 'N/A')}")
        print(f"Date: {doc.get('timestamp', 'N/A')[:10]}")
        print(f"Chunks: {len(doc.get('chunks', []))}")
        chunks = doc.get("chunks", [])
        if chunks:
            preview = chunks[0].get("text", "")[:200]
            print(f"First chunk preview: {preview}...")
        print()

## Search example 1 — Boeing corporate actions

Filter by the `Corporate Actions` tag and search for dividend and stock-split content.

In [ ]:
boeing_search = search_by_tag(
    text="Boeing corporate actions dividends and stock splits",
    tag="Corporate Actions",
)
print_search_results(boeing_search)

## Search example 2 — Brazil macro analysis

Filter by the `Brazil` tag and search for macroeconomic outlook content.

In [ ]:
brazil_search = search_by_tag(
    text="Brazil macroeconomic outlook inflation and GDP growth",
    tag="Brazil",
)
print_search_results(brazil_search)

## Re-running this notebook

Re-executing the upload cells creates new documents each time. That is expected for this tutorial. To avoid duplicates, skip the upload cells on subsequent runs and use the search cells only.